<a href="https://colab.research.google.com/github/ToffeeTofu/AI-Assignments/blob/main/Dating_Coach_AI_Powered_Attraction_Advisor_Sam_%26_Ranchy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. INSTALL / IMPORT LIBRARIES
# ============================================================

#Install Hugging Face transfomers library for NLP embeddings/tokenizations
#-q hides extra install output
!pip install transformers -q

#Imports pandas for working with tables/dataframes
import pandas as pd
#Imports NumPy for numerical operations and arrays
import numpy as np

#Imports a function to split data into training and testing sets
from sklearn.model_selection import train_test_split
#Imports the random forest machine learning classification model
from sklearn.ensemble import RandomForestClassifier
#imports logistic regression classification model
from sklearn.linear_model import LogisticRegression
#imports evaluation metrics for checking model performance
from sklearn.metrics import accuracy_score, classification_report
#imports a tool for filling in missing values in the dataset
from sklearn.impute import SimpleImputer
#Imports hugging face tokenizer loader for NLP/text processing
from transformers import AutoTokenizer

In [ ]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

# Upload your kaggle csv into Colab first
# Example:
# from google.colab import files
# files.upload()

#Reads the CSV dataset file into a pandas dataframe (visually understandable table for humans to see)
#encoding = 'Latin1' helps avoid text encoding errors from special characters
df = pd.read_csv("Speed Dating Data.csv", encoding='latin1')

#displays the first 5 rows of the dataset so we can preview the data
print(df.head())
#prints all column names in the dataset to understand available features
print(df.columns)

   gender  int_corr  samerace  attr_o  sinc_o  intel_o  fun_o  amb_o  shar_o  \
0       0      0.14         0     6.0     8.0      8.0    8.0    8.0     6.0   
1       0      0.54         0     7.0     8.0     10.0    7.0    7.0     5.0   
2       0      0.16         1    10.0    10.0     10.0   10.0   10.0    10.0   
3       0      0.61         0     7.0     8.0      9.0    8.0    9.0     8.0   
4       0      0.21         0     8.0     7.0      9.0    6.0    9.0     7.0   

    age  ...  fun3_1  intel3_1  amb3_1  dec  attr  sinc  intel  fun  amb  shar  
0  21.0  ...     8.0       8.0     7.0    1   6.0   9.0    7.0  7.0  6.0   5.0  
1  21.0  ...     8.0       8.0     7.0    1   7.0   8.0    7.0  8.0  5.0   6.0  
2  21.0  ...     8.0       8.0     7.0    1   5.0   8.0    9.0  8.0  5.0   7.0  
3  21.0  ...     8.0       8.0     7.0    1   7.0   6.0    8.0  7.0  6.0   8.0  
4  21.0  ...     8.0       8.0     7.0    1   5.0   6.0    7.0  7.0  6.0   6.0  

[5 rows x 31 columns]
Index(['ge

In [ ]:
# ============================================================
# 3. SELECT FEATURES
# ============================================================

#creates a list of input features (columns) the model will use to make predictions
features = [
    'gender', # gender of the participant
    'age', # age of the participant
    'race', # race/ethnicity of the participant
    'samerace', # whether participant and partner are the same race
    'int_corr', # similarity/correlation of interests between participant and partner
    'imprace', # importance of race similarity to the participant
    'imprelig', # importance of religion similarity to the participant

    # What the participant wants in a partner
    'attr1_1', # importance of attractiveness in a partner
    'sinc1_1', # importance of sincerity in a partner
    'intel1_1', # importance of intelligence in a partner
    'fun1_1', # importance of fun/funny personality in a partner
    'amb1_1', # importance of ambition in a partner
    'shar1_1', # importance of shared interests in a partner

    # Participant self-ratings
    'attr3_1', # participant self-rated attractiveness
    'sinc3_1', # participant self-rated sincerity
    'intel3_1', # participant self-rated intelligence
    'fun3_1', # participant self-rated fun/funny personality
    'amb3_1', # participant self-rated ambition

    # Ratings participant gave their partner
    'attr', # participant's rating of partner attractiveness
    'sinc', # participant's rating of partner sincerity
    'intel', # participant's rating of partner intelligence
    'fun', # participant's rating of partner fun/funny personality
    'amb', # participant's rating of partner ambition
    'shar', # participant's rating of shared interests with partner

    # Ratings the partner gave the participant
    'attr_o', # partner's rating of participant attractiveness
    'sinc_o', # partner's rating of participant sincerity
    'intel_o', # partner's rating of participant intelligence
    'fun_o', # partner's rating of participant fun/funny personality
    'amb_o', # partner's rating of participant ambition
    'shar_o', # partner's rating of shared interests with participant
]

#Sets the target variable the model is trying to predict
#decision represents whether the participant wanted another date
target = 'dec'

In [ ]:
# ============================================================
# 4. CLEAN DATA
# ============================================================

# Keep only needed columns
# .copy() creates a separate dataframe to avoid modifying the original dataset
data = df[features + [target]].copy()

# Creates an imputer object to handle missing values
# strategy = 'mean' replaces missing values with the average of each column
imputer = SimpleImputer(strategy='mean')
# Features
# Applies the inputer to the feature columns
# fit_trasnform = column averages and fills missing values
# X = stores the cleaned feature data used for training
X = imputer.fit_transform(data[features])

# Target
# Extracts the target volumn
# .values = converts the column into a NumPy array
# .ravel = flattens it into a 1D array which is required by sklearn models
y = data[target].values.ravel()

# Convert non-numeric columns if necessary
# (field/race sometimes messy so we skip for simplicity)


In [ ]:
# ============================================================
# 5. TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, #input feature data
    y, #targets labels (what we want to predict)
    test_size=0.2, #20% of the data for resting and 80% for training
    random_state=42 #sets a fixed random seed so results are reproducible
)

In [ ]:
# ============================================================
# 6. RANDOM FOREST MODEL - captures the nonlinear relationships
# ============================================================

#We build, train, and evaluate the Random Forest model for the dating coach system
#model uses many decision trees to learn from the dating dataset & predicts whether a dating interaction will lead to a positive outcome
rf_model = RandomForestClassifier(
    #number of decision trees used in forest
    #more trees usually improve accuracy & reduces overfitting but increase computation time
    n_estimators=100,
    #sets a fixed random seed so results stay consistent on each run
    random_state=42
)

#trains the random forest model using the training data
#Ensures the model produces consistent results every time the notebook runs by fixing the randomness
rf_model.fit(X_train, y_train)

# Predictions - trained model to predict outcomes on the test data (unseen) <-- checks how well the model generalizes to new examples
rf_preds = rf_model.predict(X_test)

print("\n===== RANDOM FOREST RESULTS =====")

# Calculates and prints the model accuracy
# Accuracy = percentage of correct predictions
print("Accuracy:", accuracy_score(y_test, rf_preds)) #measures model performance by calculating the % of correct predictions


===== RANDOM FOREST RESULTS =====
Accuracy: 0.7983293556085919


In [ ]:
# ============================================================
# 7. LOGISTIC REGRESSION MODEL - creates a second ML model to compare against the Random Forest one bc simpler
# ============================================================

#Creates a logistic regression classification model
lr_model = LogisticRegression(max_iter=1000) #increases max number of training iterations to help with convergence

# Trains the Logistic Regression model using the training data
lr_model.fit(X_train, y_train)


# Uses the trained model to predict outcomes on the test data
lr_preds = lr_model.predict(X_test)

print("\n===== LOGISTIC REGRESSION RESULTS =====")

# Calculates and prints the model accuracy
# Accuracy = percentage of correct predictions
print("Accuracy:", accuracy_score(y_test, lr_preds))


===== LOGISTIC REGRESSION RESULTS =====
Accuracy: 0.7553699284009546


In [ ]:
# ============================================================
# 8. FEATURE IMPORTANCE for when the Random Forestmodel made dating predictions
# ============================================================

#Creates a table showing each feature name and its importance score from Random Forest model
importance_df = pd.DataFrame({
    'Feature': features, #column containing the feature names
    #higher values mean the feature has a bigger impact on predictions
    #lower values means the features mattered less
    'Importance': rf_model.feature_importances_
})

# Sorts the dataframe by importance score from highest to lowest importance
importance_df = importance_df.sort_values(
    by='Importance',  # Sort using the 'Importance' column
    ascending=False # descending order so most important features appear first
)

#Prints a section header for feature impotance results
print("\n===== MOST IMPORTANT FEATURES =====")
print(importance_df.head(10)) #displays the top 10 most important features

#Ex: in the output attractiveness shows up first meaning that it had the biggest effect on match outcomes
#DOES not mean that this particular feature importance causes dating success - rather it means that the model relies heavily on the variable when making predictions


===== MOST IMPORTANT FEATURES =====
     Feature  Importance
18      attr    0.153513
21       fun    0.095575
23      shar    0.074496
4   int_corr    0.046022
12   shar1_1    0.032847
1        age    0.030992
7    attr1_1    0.030732
9   intel1_1    0.029986
19      sinc    0.029443
29    shar_o    0.028496


In [ ]:
# ============================================================
# 9. TOKENIZATION EXAMPLE (NLP COMPONENT) - converts natural language input into tokens & numerical IDs that the AI models can process to understand & analyze real user text questions
# ============================================================

print("\n===== TOKENIZATION EXAMPLE =====")

# Loads a pre-trained BERT tokenizer (uncased = ignores capitalization)
# BERT is a powerful NLP model trained on massive amounts of text
# bert-base-uncased = standard-sized BERT model, uncased = ignores capitilization
# This converts text into tokens the model can understand
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Example user input (a dating-related question)
sample_prompt = "Why am I not getting matches?"

# Splits the sentence into subword tokens used by the BERT model
tokens = tokenizer.tokenize(sample_prompt)

# Converts each token into its corresponding numerical ID
# (these IDs are what ML models actually process bc AI doesn't understand raw sentences)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

#Prints the original input text
print("Original Prompt:")
print(sample_prompt)

#Prints the tokenized version of the text
print("\nTokens:")
print(tokens)

#Prints the numeric token IDs corresponding to each token
print("\nToken IDs:")
print(token_ids)


===== TOKENIZATION EXAMPLE =====
Original Prompt:
Why am I not getting matches?

Tokens:
['why', 'am', 'i', 'not', 'getting', 'matches', '?']

Token IDs:
[2339, 2572, 1045, 2025, 2893, 3503, 1029]


In [ ]:
# ============================================================
# 10. SAFE INPUT HELPER - ensures the chatbot only accepts valid numerical inputs within allowed ranges
# ============================================================

#Defines a reusable function for safely collecting user input
#prompt = text shown to user
#min_val = min allowed value
#max_val = max allowed value
#allowed = specific accepted values only
def get_valid_input(prompt, min_val=None, max_val=None, allowed=None):

    while True: #infinite loop bc we want the fxn to keep asking until the user enter a valid answer

        try:
            value = float(input(prompt)) #gets user input & convert it to a number

            # Check exact allowed values
            if allowed is not None:
                if value not in allowed:
                    print(f"Invalid input. Allowed values: {allowed}")
                    continue

            # Check min/max range
            if min_val is not None and value < min_val: #prevents 0 and negative numbers
                print(f"Value must be at least {min_val}.")
                continue

            if max_val is not None and value > max_val: #checks the max allowed value + prevents rating over 10
                print(f"Value must be at most {max_val}.")
                continue

            return value

        except ValueError: #prevents crash
            print("Please enter a valid number.")

In [ ]:
# ============================================================
# 11. USER PROFILE INPUT - collect structured dating info, validates inputs, and organizes the data into the exact format needed by the ML model
# ============================================================

#Defines a fxn that collects all user profile information needed
def get_user_profile():

    #Prints instructions for the user
    print("\n===== ENTER YOUR DATING PROFILE =====")
    print("Most ratings use a 1-10 scale.")
    print("Preference values should roughly add to 100.\n")

    #Displays race encoding values used in the dataset
    print("Race Codes:")
    print("1=Black  2=White  3=Latino")
    print("4=Asian  5=Native American  6=Other\n")

    print("\n--- BASIC INFO ---")

    #Gets gendr input and restricts values to 0 or 1
    gender = get_valid_input(
        "Gender (0=female, 1=male): ",
        allowed=[0, 1]
    )

    #Gets age input between 18 and 100
    age = get_valid_input(
        "Age (18-100): ",
        min_val=18,
        max_val=100
    )

    #Gets race category using encoded numeric values
    race = get_valid_input(
        "Race (1=Black, 2=White, 3=Latino, 4=Asian, 5=Native American, 6=Other): ",
        allowed=[1, 2, 3, 4, 5, 6]
    )

    #Determines whether the user prefers same-race matches
    samerace = get_valid_input(
        "Prefer same race? (0=no, 1=yes): ",
        allowed=[0, 1]
    )

    #Measures preferences for shared interests/compatibility
    int_corr = get_valid_input(
        "Interest similarity preference (0-10): ",
        min_val=0,
        max_val=10
    )

    #Importance of race in dating preferences
    imprace = get_valid_input(
        "Importance of race (1-10): ",
        min_val=1,
        max_val=10
    )

    #Importance of religion in dating preferences
    imprelig = get_valid_input(
        "Importance of religion (1-10): ",
        min_val=1,
        max_val=10
    )

    print("\n--- IDEAL PARTNER TRAITS ---")
    print("Use percentages totaling roughly 100.\n")

    #% importance of attractiveness in a partner
    attr1_1 = get_valid_input(
        "Attractiveness importance (%): ",
        min_val=0,
        max_val=100
    )

    #% importance of sincerity in a partner
    sinc1_1 = get_valid_input(
        "Sincerity importance (%): ",
        min_val=0,
        max_val=100
    )

    #% importance of intelligence in a partner
    intel1_1 = get_valid_input(
        "Intelligence importance (%): ",
        min_val=0,
        max_val=100
    )

    #% importance of humor/fun in a partner
    fun1_1 = get_valid_input(
        "Fun/humor importance (%): ",
        min_val=0,
        max_val=100
    )

    #% importance of ambition in a partner
    amb1_1 = get_valid_input(
        "Ambition importance (%): ",
        min_val=0,
        max_val=100
    )

    #% importance of shared interests in a partner
    shar1_1 = get_valid_input(
        "Shared interests importance (%): ",
        min_val=0,
        max_val=100
    )

    print("\n--- SELF RATINGS ---")

    # User self-rating for attractiveness
    attr3_1 = get_valid_input(
        "Your attractiveness (1-10): ",
        min_val=1,
        max_val=10
    )

    # User self-rating for sincerity
    sinc3_1 = get_valid_input(
        "Your sincerity (1-10): ",
        min_val=1,
        max_val=10
    )

    # User self-rating for intelligence
    intel3_1 = get_valid_input(
        "Your intelligence (1-10): ",
        min_val=1,
        max_val=10
    )

    # User self-rating for humor
    fun3_1 = get_valid_input(
        "Your fun/humor (1-10): ",
        min_val=1,
        max_val=10
    )

    # User self-rating for ambition
    amb3_1 = get_valid_input(
        "Your ambition (1-10): ",
        min_val=1,
        max_val=10
    )

    # Optional: check total preference percentages
    total_preferences = (
        attr1_1 + sinc1_1 + intel1_1 +
        fun1_1 + amb1_1 + shar1_1
    )

    # Warns user if percentages do not total approximately 100
    if total_preferences != 100:
        print(f"\nNote: Your preference total is {total_preferences}, not 100.")

    # Creates a dictionary containing all user profile data
    profile = {
        'gender': gender,
        'age': age,
        'race': race,
        'samerace': samerace,
        'int_corr': int_corr,
        'imprace': imprace,
        'imprelig': imprelig,

        'attr1_1': attr1_1,
        'sinc1_1': sinc1_1,
        'intel1_1': intel1_1,
        'fun1_1': fun1_1,
        'amb1_1': amb1_1,
        'shar1_1': shar1_1,

        'attr3_1': attr3_1,
        'sinc3_1': sinc3_1,
        'intel3_1': intel3_1,
        'fun3_1': fun3_1,
        'amb3_1': amb3_1
    }

    # Returns the completed profile dictionary
    return profile

    #converts technical dataset feature names into understandable labels for chatbot feedback and explanations

In [ ]:
feature_labels = {

    'attr1_1': 'preference for attractiveness',
    'sinc1_1': 'preference for sincerity',
    'intel1_1': 'preference for intelligence',
    'fun1_1': 'preference for humor',
    'amb1_1': 'preference for ambition',
    'shar1_1': 'preference for shared interests',

    'attr3_1': 'attractiveness',
    'sinc3_1': 'sincerity',
    'intel3_1': 'intelligence',
    'fun3_1': 'humor',
    'amb3_1': 'ambition',

    'int_corr': 'shared interests compatibility',
    'imprace': 'importance of race',
    'imprelig': 'importance of religion'
}

In [ ]:
# ============================================================
# 12. PERSONALIZED FEEDBACK GENERATION
# ============================================================

#Defines a function that generates personalized dating feedback based on the user's profile and ML feature importance
def generate_feedback(profile):

    #Imports Python's random module for randomized response selection
    import random

    #Stores generated feedback messages
    feedback = []

    # Convert user profile into dataframe
    user_df = pd.DataFrame([profile])

    # Add missing feature columns if needed (and fills with NaN values)
    for col in features:
        if col not in user_df.columns:
            user_df[col] = np.nan

    # Reorders columns to match the exact feature order used during model training
    user_df = user_df[features]

    # Applies the trained imputer to fill missing values using the same preprocessing used during training
    user_df_imputed = imputer.transform(user_df)

    # Retrieves feature importance scores from the trained Random Forest model
    importances = rf_model.feature_importances_

    # Dictionary that converts technical dataset feature names into human-readable labels for chatbot responses
    feature_labels = {

        # Partner preferences
        'attr1_1': 'preference for attractiveness',
        'sinc1_1': 'preference for sincerity',
        'intel1_1': 'preference for intelligence',
        'fun1_1': 'preference for humor',
        'amb1_1': 'preference for ambition',
        'shar1_1': 'interest in shared hobbies and activities',

        # Self ratings
        'attr3_1': 'confidence and attractiveness',
        'sinc3_1': 'sincerity',
        'intel3_1': 'intelligence',
        'fun3_1': 'sense of humor',
        'amb3_1': 'ambition and drive',

        # Other traits
        'int_corr': 'ability to connect through shared interests',
        'imprace': 'focus on racial preferences',
        'imprelig': 'focus on religious preferences'
    }

    # Positive feedback templates
    positive_templates = [
        "One of your biggest strengths is your {}.",
        "Your {} stands out as a strong positive trait.",
        "Your {} is likely attractive to the type of partner you want.",
        "Your {} could help you form strong connections.",
        "People with similar dating preferences often benefit from strong {}."
    ]

    # Improvement feedback templates
    negative_templates = [
        "One area you could improve is your {}.",
        "Developing your {} could improve your matches.",
        "Your {} may be holding back your dating success.",
        "Improving your {} may help you connect more easily.",
        "People with similar preferences often succeed by improving their {}."
    ]

    # Store feature analysis
    feature_analysis = []

    #Loops through all model features
    for i, feature in enumerate(features):

        # Retrieves the user's value for this feature
        value = user_df.iloc[0][feature]
        # Retrieves the Random Forest importance score
        importance = importances[i]

         # Stores feature name, user value, and importance score
        feature_analysis.append((feature, value, importance))

    # Sort features by importance
    feature_analysis.sort(
        key=lambda x: x[2],
        reverse=True
    )

    # Generate feedback using only the top 7 most important features
    for feature, value, importance in feature_analysis[:7]:

        # Only process features with readable labels
        if feature in feature_labels:

            # Converts technical feature name into readable text
            readable = feature_labels[feature]

            # Strong traits
            if value >= 7:

                feedback.append(
                    random.choice(positive_templates).format(readable)
                )

            # Weak traits
            elif value <= 3:

                feedback.append(
                    random.choice(negative_templates).format(readable)
                )

    # Prevent empty feedback if no major strengths or weaknesses are found
    if len(feedback) == 0:

        feedback.append(
            "Your profile appears fairly balanced overall."
        )

    # Returns the list of generated feedback messages
    return feedback

In [ ]:
# ============================================================
# 12. MATCH PREDICTION - convert user input into model-ready data and use the trained AI model to predict match likelihood
# ============================================================

#Defines a fxn that takes a user's profile and predicts whether they will get a match using the trained model
def predict_match(profile):

    #Converts the user's profile dictionary into a pandas DataFrame
    #The model expects tabular dataframe input
    user_df = pd.DataFrame([profile])

    # Add missing columns automatically
    for col in features:
        if col not in user_df.columns:
            user_df[col] = np.nan

    # Match training feature order
    user_df = user_df[features]

    # Fill missing values
    user_df = imputer.transform(user_df)

    #Uses the Random Forest model to predict whether the dating outcome is positive or negative
    prediction = rf_model.predict(user_df)[0]
    #Retrieves probability of a positive match outcome and values range from 0 to 1
    probability = rf_model.predict_proba(user_df)[0][1]

    #returns both prediction and probability score
    return prediction, probability

In [ ]:
# ============================================================
# 13. CHATBOT RESPONSE
# ============================================================

#Defines the main chatbot function that runs the full user experience
def chatbot():

    print("\n===================================")
    print("AI RELATIONSHIP COACH CHATBOT")
    print("===================================")

    #collects user input profile through interactive prompts
    user_profile = get_user_profile()

     # Generates strengths and weaknesses based on rule-based logic
    feedback = generate_feedback(user_profile)

    # Uses trained ML model to predict match outcome and probability of positive dating success
    prediction, probability = predict_match(user_profile)

    print("\n========== PERSONALIZED FEEDBACK ==========")

    #prints each generated feedback message
    if feedback:
      for item in feedback:
        print("-", item)
    #fallback message if no strong patterns are detected
    else:
      print("No strong patterns detected.")

    print("\n========== MATCH PREDICTION ==========")

    # Displays probability of being liked in percentage form
    print("Likelihood partner will like you:",
          round(probability * 100, 2), "%")

    #Displays prediction interpreation
    if prediction == 1:
        print("Prediction: Positive dating outcome likely")
    else:
        print("Prediction: Lower likelihood of positive match")

    print("\n========== MODEL ADVICE ==========")

    #high compatibility advice
    if probability >= 0.75:
      print("- Your profile aligns very well with your preferred partner traits.")
      print("- Your strongest traits appear highly compatible with your dating preferences.")

    #moderate compatibility advice
    elif probability >= 0.5:
      print("- Your profile shows moderate compatibility.")
      print("- Shared interests and communication style may strongly influence success.")

    #lower compatibility advice
    else:
      print("- Your current preferences and self-ratings show weaker alignment patterns.")
      print("- People with similar preferences often benefit from emphasizing shared interests, sincerity, or communication.")

In [ ]:
# ============================================================
# 14. RUN CHATBOT
# ============================================================

chatbot()


AI RELATIONSHIP COACH CHATBOT

===== ENTER YOUR DATING PROFILE =====
Most ratings use a 1-10 scale.
Preference values should roughly add to 100.

Race Codes:
1=Black  2=White  3=Latino
4=Asian  5=Native American  6=Other


--- BASIC INFO ---
Gender (0=female, 1=male): 0
Age (18-100): 67
Race (1=Black, 2=White, 3=Latino, 4=Asian, 5=Native American, 6=Other): 4
Prefer same race? (0=no, 1=yes): 1
Interest similarity preference (0-10): 5
Importance of race (1-10): 9
Importance of religion (1-10): 9

--- IDEAL PARTNER TRAITS ---
Use percentages totaling roughly 100.

Attractiveness importance (%): 10
Sincerity importance (%): 10
Intelligence importance (%): 10
Fun/humor importance (%): 10
Ambition importance (%): 10
Shared interests importance (%): 50

--- SELF RATINGS ---
Your attractiveness (1-10): 7
Your sincerity (1-10): 9
Your intelligence (1-10): 8
Your fun/humor (1-10): 8
Your ambition (1-10): 6

========== PERSONALIZED FEEDBACK ==========
- One of your biggest strengths is your int